In [32]:
!pip install -q langchain
!pip install -q langchain_community
!pip install -q pypdf
!pip install -q langchain-groq
!pip install -q langchain-huggingface sentence-transformers
!pip install -q langchain-chroma
!pip install rich

In [33]:
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader # We can have other types of loader, too!
from langchain_text_splitters import CharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from rich.console import Console
from rich.markdown import Markdown

In [34]:

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=userdata.get("GROQ_API_KEY")
)
#embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2" )
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

console = Console()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [46]:
person_name = "Ramtin Hamavar"
question = "Where did he get his phd?"

In [47]:
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert on humans!"),
    ("human", "answer the following question about {person_name}, here is some context\n{retrieved_context}"),
    ("ai", "sure!"),
    ("human", "{question}")
])

chain = prompt_template | llm

In [48]:
pdf_files = ["1.pdf", "2.pdf", "3.pdf"]

pdf_data = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(pdf_file)
    pdf_data.extend(loader.load_and_split())

In [ ]:
console.print(Markdown(pdf_data[0].page_content))

In [49]:
response = chain.invoke({"person_name":person_name, "retrieved_context":pdf_data[2], "question":question})

In [40]:
console.print(Markdown(response.content))

I’m sorry, but I don’t have any information about Ramtin Hamavar’s research interests in the material you provided 
or in my current knowledge base. If you can share more context—such as his field, institution, or any publications 
he’s authored—I’ll do my best to help you identify his research areas.

In [41]:
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=200,
                                      length_function=len, is_separator_regex=False)
texts = text_splitter.split_documents(pdf_data)


vectorstore = Chroma.from_documents(
    documents=texts,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

In [50]:
docs = retriever.invoke(person_name)

In [51]:
print(docs[0])

page_content='1 
 
 Ramtin Hamavar  
                                                                                                                                                 Tehran, Iran 
                                                                                                                                                 +98 (917) 192-7587 
                                                                                                                                                 ramtinhamavar@gmail.com 
                                                                                                                                                 github.com/ramtinhamavar 
                                                                                                                                                 linkedin.com/in/ramtin-hamavar-397b5384/ 
   Education 
 
2020–2025 Ph.D. in Electrical Engineering  
Tarbiat Modares University, Faculty of Electrical a

In [52]:
response = chain.invoke({"person_name":person_name, "retrieved_context":docs, "question":question})

In [53]:
console.print(Markdown(response.content))

Ramtin Hamavar earned his Ph.D. in Electrical Engineering (2020 – 2025) at Tarbiat Modares University, specifically
within the Faculty of Electrical and Computer Engineering.